In [1]:
# ────────────────────────────────────────────────────────────────
# 0. Quick environment check
# ────────────────────────────────────────────────────────────────
import torch

print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
print("Device count:", torch.cuda.device_count())
print("Current device:", torch.cuda.current_device() if torch.cuda.is_available() else "cpu")
print("Device name:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")
print("CUDA in PyTorch:", torch.version.cuda)
print("cuDNN:", torch.backends.cudnn.version() if torch.backends.cudnn.is_available() else "N/A")

# !nvidia-smi   # run this in terminal or as cell magic if in Jupyter

torch.cuda.empty_cache()

PyTorch: 2.10.0+cu130
CUDA available: True
Device count: 1
Current device: 0
Device name: NVIDIA RTX A6000
CUDA in PyTorch: 13.0
cuDNN: 91200


In [2]:
# ────────────────────────────────────────────────────────────────
# 1. Imports
# ────────────────────────────────────────────────────────────────
import numpy as np
from tqdm.auto import tqdm
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    Trainer,
    TrainingArguments,
    EarlyStoppingCallback,
    DataCollatorWithPadding
)
from datasets import load_dataset
from sklearn.metrics import accuracy_score, precision_recall_fscore_support

c:\gyanendra\non_aug\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
# ────────────────────────────────────────────────────────────────
# 2. Load your local stratified CSVs
# ────────────────────────────────────────────────────────────────
dataset = load_dataset(
    "csv",
    data_files={
        "train": "./train.csv",
        "validation": "./val.csv",
        "test": "./test.csv",
    }
)

print(f"Train examples: {len(dataset['train']):,}")
print(f"Val examples:   {len(dataset['validation']):,}")
print(f"Test examples:  {len(dataset['test']):,}")

Train examples: 2,344,467
Val examples:   293,058
Test examples:  293,059


In [4]:
# from huggingface_hub import login
# login() 

In [5]:
# ────────────────────────────────────────────────────────────────
# 3. Tokenizer + efficient tokenization (optimized local version)
# ────────────────────────────────────────────────────────────────
from transformers import AutoTokenizer, DataCollatorWithPadding

tokenizer = AutoTokenizer.from_pretrained("roberta-base")

# Optional: Print a few examples to understand real lengths (very useful!)
print("Checking token lengths of first 5 train examples...")
for i in range(5):
    tokens = tokenizer(dataset["train"][i]["text"], add_special_tokens=True)
    print(f"Example {i}: {len(tokens['input_ids'])} tokens")


MAX_LENGTH = 256   # ← change to 256 or 320 if most examples are shorter

def preprocess(examples, tokenizer):          # ← add tokenizer as argument
    tokenized = tokenizer(
        examples["text"],
        truncation=True,
        max_length=256,
        add_special_tokens=True,
        return_attention_mask=True,
        return_token_type_ids=False,
    )
    tokenized["labels"] = examples["label"]
    return tokenized


# ── changed part ──
tokenized_datasets = dataset.map(
    preprocess,
    batched=True,
    batch_size=1000,
    num_proc=10,  # can stay >1
    desc="Tokenizing",
    remove_columns=dataset["train"].column_names,
    fn_kwargs={"tokenizer": tokenizer},     # ← this is the key line
)

# Dynamic padding → perfect choice
data_collator = DataCollatorWithPadding(
    tokenizer=tokenizer,
    padding=True,                  # or 'longest' — same effect here
    max_length=MAX_LENGTH          # helps trainer know bound
)

print("Tokenization done. Sample tokenized item keys:", tokenized_datasets["train"].features.keys())

Token indices sequence length is longer than the specified maximum sequence length for this model (757 > 512). Running this sequence through the model will result in indexing errors


Checking token lengths of first 5 train examples...
Example 0: 131 tokens
Example 1: 757 tokens
Example 2: 215 tokens
Example 3: 822 tokens
Example 4: 785 tokens


Tokenizing (num_proc=10): 100%|██████████| 293059/293059 [01:12<00:00, 4041.34 examples/s]

Tokenization done. Sample tokenized item keys: dict_keys(['input_ids', 'attention_mask', 'labels'])


In [6]:
# ────────────────────────────────────────────────────────────────
# 4. Metrics
# ────────────────────────────────────────────────────────────────
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    precision, recall, f1, _ = precision_recall_fscore_support(labels, preds, average="binary")
    acc = accuracy_score(labels, preds)
    return {
        "accuracy": round(acc, 4),
        "precision": round(precision, 4),
        "recall": round(recall, 4),
        "f1": round(f1, 4)
    }

In [7]:
# ────────────────────────────────────────────────────────────────
# 5. TrainingArguments – tuned for local GPU
# ────────────────────────────────────────────────────────────────
training_args = TrainingArguments(
    output_dir="./roberta-fake-news-local",
    num_train_epochs=4,                     # early stopping usually exits earlier
    per_device_train_batch_size=32,         # start here – increase to 24/32 if you have ≥10 GB VRAM
    per_device_eval_batch_size=32,
    gradient_accumulation_steps=2,          # effective batch = 16×2×(1 or more GPUs) = 32+
    fp16=True,                              # huge speedup on Ampere+ GPUs (RTX 30/40 series)
    # bf16=True,                            # use instead of fp16 if you have Ampere+ and want better numerics
    eval_strategy="epoch",                         # evaluate at end of each epoch
    # eval_steps=2000,                         # less frequent eval = faster wall time
    save_strategy="epoch",                         # save at end of each epoch
    # save_steps=2000,
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="eval_f1",
    greater_is_better=True,
    learning_rate=2e-5,
    weight_decay=0.01,
    warmup_ratio=0.1,
    logging_steps=50,
    report_to="none",                       # no wandb / tensorboard clutter
    # max_steps=1500,                       # safety cap – remove if not needed
)

warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


In [8]:
# ────────────────────────────────────────────────────────────────
# 6. Trainer setup
# ────────────────────────────────────────────────────────────────
model = AutoModelForSequenceClassification.from_pretrained(
    "roberta-base",
    num_labels=2
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["validation"],
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    callbacks=[
        EarlyStoppingCallback(
            early_stopping_patience=3,
            early_stopping_threshold=0.001
        )
    ],
)

# Quick sanity check
print("Device:", trainer.args.device)
print("GPUs:", trainer.args.n_gpu)
print("Effective batch size:", 
      training_args.per_device_train_batch_size * 
      training_args.gradient_accumulation_steps * 
      max(1, trainer.args.n_gpu))

Loading weights: 100%|██████████| 197/197 [00:00<00:00, 698.40it/s, Materializing param=roberta.encoder.layer.11.output.dense.weight]              
RobertaForSequenceClassification LOAD REPORT from: roberta-base
Key                             | Status     | 
--------------------------------+------------+-
lm_head.bias                    | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
classifier.out_proj.weight      | MISSING    | 
classifier.dense.bias           | MISSING    | 
classifier.out_proj.bias        | MISSING    | 
classifier.dense.weight         | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Conside

Device: cuda:0
GPUs: 1
Effective batch size: 64


# dont close this window just minimize it 

name - gyanendra

In [9]:
# ────────────────────────────────────────────────────────────────
# 7. Train
# ────────────────────────────────────────────────────────────────
print("\n" + "═"*70)
print("Starting local fine-tuning of RoBERTa-base")
print("═"*70 + "\n")

trainer.train()


══════════════════════════════════════════════════════════════════════
Starting local fine-tuning of RoBERTa-base
══════════════════════════════════════════════════════════════════════



c:\gyanendra\non_aug\.venv\Lib\site-packages\transformers\tokenization_utils_base.py:2299: UserWarning: `max_length` is ignored when `padding`=`True` and there is no truncation strategy. To pad to max length, use `padding='max_length'`.
  warnings.warn(


Epoch,Training Loss,Validation Loss,Accuracy,Precision,Recall,F1
1,0.023015,0.011818,0.997600,0.998200,0.994800,0.996500
2,0.011788,0.009075,0.998000,0.996800,0.997500,0.997100
3,0.010746,0.008308,0.998300,0.997200,0.997900,0.997600
4,0.000102,0.007791,0.998600,0.998700,0.997300,0.998000


Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  1.52it/s]
c:\gyanendra\non_aug\.venv\Lib\site-packages\transformers\tokenization_utils_base.py:2299: UserWarning: `max_length` is ignored when `padding`=`True` and there is no truncation strategy. To pad to max length, use `padding='max_length'`.
  warnings.warn(
Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  1.60it/s]
c:\gyanendra\non_aug\.venv\Lib\site-packages\transformers\tokenization_utils_base.py:2299: UserWarning: `max_length` is ignored when `padding`=`True` and there is no truncation strategy. To pad to max length, use `padding='max_length'`.
  warnings.warn(
Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  1.67it/s]
c:\gyanendra\non_aug\.venv\Lib\site-packages\transformers\tokenization_utils_base.py:2299: UserWarning: `max_length` is ignored when `padding`=`True` and there is no truncation strategy. To pad to max length, use `padding='max_length'`.
  warnings.warn(
Writing model shards: 100%|███████

TrainOutput(global_step=146532, training_loss=0.022632566853374406, metrics={'train_runtime': 38949.6942, 'train_samples_per_second': 240.769, 'train_steps_per_second': 3.762, 'total_flos': 1.2337103732533862e+18, 'train_loss': 0.022632566853374406, 'epoch': 4.0})

# dont close this window just minimize it 

In [10]:
# ────────────────────────────────────────────────────────────────
# 8. Test set evaluation
# ────────────────────────────────────────────────────────────────
print("\nFinal test evaluation...")
test_results = trainer.evaluate(tokenized_datasets["test"])
print("\nTest results:")
for k, v in test_results.items():
    if k.startswith("eval_"):
        print(f"{k.replace('eval_', '').ljust(12)} : {v:.4f}")

c:\gyanendra\non_aug\.venv\Lib\site-packages\transformers\tokenization_utils_base.py:2299: UserWarning: `max_length` is ignored when `padding`=`True` and there is no truncation strategy. To pad to max length, use `padding='max_length'`.
  warnings.warn(



Final test evaluation...



Test results:
loss         : 0.0078
accuracy     : 0.9986
precision    : 0.9989
recall       : 0.9971
f1           : 0.9980
runtime      : 385.6032
samples_per_second : 760.0010
steps_per_second : 23.7520


In [11]:
# ────────────────────────────────────────────────────────────────
# 9. Save best model
# ────────────────────────────────────────────────────────────────
save_path = "./roberta-fake-news-best-local"
trainer.save_model(save_path)
tokenizer.save_pretrained(save_path)
print(f"Model saved → {save_path}")

Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  1.60it/s]

Model saved → ./roberta-fake-news-best-local
